# Can we predict late delivery?
Follows `protocol.yaml`: temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, choice by mean CV ROC AUC, one holdout score.

**Feature decision:** `claim_paid_eur` is compensation paid on a claim settled *after* delivery (about 90% of rows with a claim are late, and no on-time row has one). It is not known at prediction time, so it is excluded as leakage. `consignment` and `dispatched_on` are excluded per the protocol (dispatch time is used only for ordering/splitting).

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==proto['dataset']['rows'] and len(hold_idx)==proto['split']['holdout_rows']
df['t'] = pd.to_datetime(df['dispatched_on'])
train = df[~df.index.isin(hold_idx)].sort_values('t', kind='stable')   # index keeps original row positions
hold = df.loc[hold_idx]
print(len(train), len(hold), train.t.max(), hold.t.min())

In [ ]:
NUM = ['distance_km','weight_kg','parcels']; CAT = ['carrier']
FEATURES = NUM + CAT
y_tr = train['late_delivery'].to_numpy(); X_tr = train[FEATURES]
print(train.late_delivery.mean()); print(train.groupby('carrier').late_delivery.agg(['mean','size']))

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
CANDIDATES = {
 'baseline_prior': lambda: Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': lambda: Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': lambda: Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest': lambda: Pipeline([('p',prep(False)),('m',RandomForestClassifier(300,min_samples_leaf=20,random_state=0))]),
 'hist_gb': lambda: Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(CANDIDATES))

## Walk-forward cross-validation on training rows (preprocessing fitted inside each fold)

In [ ]:
folds = list(TimeSeriesSplit(n_splits=proto['cv']['n_splits']).split(X_tr))
pos = train.index.to_numpy()   # 0-based positions in data.csv
cv_folds = [[pos[a].tolist(), pos[b].tolist()] for a,b in folds]   # [train rows, test rows]
cv_scores = {}
for name, mk in CANDIDATES.items():
    s=[]
    for a,b in folds:
        m = mk().fit(X_tr.iloc[a], y_tr[a])
        s.append(float(roc_auc_score(y_tr[b], m.predict_proba(X_tr.iloc[b])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen = cv['mean'].idxmax(); print('chosen:', chosen)

## Refit on all training rows, score once on holdout

In [ ]:
final = CANDIDATES[chosen]().fit(X_tr, y_tr)
holdout_auc = float(roc_auc_score(hold['late_delivery'], final.predict_proba(hold[FEATURES])[:,1]))
print('Holdout ROC AUC:', round(holdout_auc,4), '(holdout positive rate: %.3f)' % hold.late_delivery.mean())

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(CANDIDATES),
 'cv_folds': cv_folds,
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
The printed CV table and holdout ROC AUC above are the result. Without the post-outcome `claim_paid_eur` column, predictability comes only from dispatch-time attributes (carrier, distance, weight, parcels); the holdout AUC is the honest estimate of how well late delivery can be predicted at dispatch.